# Naive Bayes Classifier

## Learning Objectives
- Understand the **naïve conditional independence** assumption and when it is reasonable
- Derive the MAP decision rule from Bayes' theorem in log-space
- Compute **Gaussian NB** MLE estimates for $\mu_{kj}$ and $\sigma^2_{kj}$
- Apply the **log-sum trick** to avoid numerical underflow during scoring
- Implement a vectorised `fit / predict_log_proba / predict` in NumPy

## Problem Statement

Given a labelled training set $\{(x^{(i)}, y^{(i)})\}_{i=1}^{m}$ with $x^{(i)} \in \mathbb{R}^n$ and $y^{(i)} \in \{0, \ldots, K{-}1\}$, learn a classifier that assigns the most probable class to any new input $x$.

---

### Generative vs Discriminative

| Approach | Models | Examples |
|---|---|---|
| **Discriminative** | $P(y \mid x)$ directly | Logistic Regression, SVM |
| **Generative** | $P(x, y) = P(x \mid y)\,P(y)$ | Naive Bayes, GDA |

Naive Bayes is a **generative** model — it learns the joint distribution and recovers $P(y \mid x)$ via Bayes' theorem.

---

### Model Assumptions

| Component | Distribution | Parameters |
|---|---|---|
| **Prior** $P(y=k)$ | Categorical | $\phi_k = m_k / m$ |
| **Likelihood** $P(x_j \mid y=k)$ | Gaussian (GNB) | $\mu_{kj},\; \sigma^2_{kj}$ |
| **Independence** | $P(x \mid y=k) = \prod_{j=1}^{n} P(x_j \mid y=k)$ | — |
| **Posterior** $P(y \mid x)$ | via Bayes | — |

## The Naïve Bayes Assumption

The **naïve conditional independence** assumption states that all features $x_1, x_2, \ldots, x_n$ are **conditionally independent** given the class label $y$:

$$P(x_1, x_2, \ldots, x_n \mid y = k) = \prod_{j=1}^{n} P(x_j \mid y = k)$$

This factorisation allows the intractable joint feature likelihood to be replaced by a product of simple per-feature likelihoods, reducing the parameter count from **exponential** to **linear** in $n$.

| | Without assumption | With assumption (GNB) |
|---|---|---|
| **Parameters per class** | Full covariance: $n + \binom{n}{2}$ | Mean + variance per feature: $2n$ |
| **Example** ($n=30$, $K=2$) | $2 \times 495 = 990$ | $2 \times 60 = 120$ |
| **Tractability** | Requires large $m$ to estimate $\Sigma$ | Works even when $m \ll n$ |

> **Why "naïve"?** Real-world features are often correlated (e.g. word co-occurrences in text). The assumption is almost never exactly satisfied — yet Naive Bayes classifiers achieve surprisingly competitive accuracy in practice.

### Formal statement

For a new input $x \in \mathbb{R}^n$, the posterior is:

$$P(y=k \mid x) \propto P(y=k) \prod_{j=1}^{n} P(x_j \mid y=k)$$

where each factor $P(x_j \mid y=k)$ is modelled independently — typically as $\mathcal{N}(\mu_{kj},\, \sigma^2_{kj})$ in **Gaussian NB**.

In [ ]:
from IPython.display import SVG, display

svg = """
<svg xmlns="http://www.w3.org/2000/svg" width="680" height="320" viewBox="0 0 680 320">
  <rect width="680" height="320" fill="#fafafa" rx="8"/>
  <text x="340" y="24" text-anchor="middle" font-size="13" font-weight="bold" fill="#222">Naive Bayes &#x2014; Generative Independence Model</text>

  <!-- Training data box (top, centered) -->
  <rect x="190" y="40" width="300" height="52" rx="6" fill="#e8f4fd" stroke="#4a90d9" stroke-width="2"/>
  <text x="340" y="62" text-anchor="middle" font-size="11" font-weight="bold" fill="#4a90d9">Training Data</text>
  <text x="340" y="80" text-anchor="middle" font-size="10" fill="#333">X &#x2208; &#x211D;&#x1D39;&#xD7;&#x1D3F;,  y &#x2208; {0, &#x2026;, K&#x2212;1}</text>

  <!-- Arrows from top box to three parameter boxes -->
  <line x1="278" y1="92" x2="115" y2="130" stroke="#888" stroke-width="1.5"/>
  <polygon points="110,126 108,137 119,133" fill="#888"/>

  <line x1="340" y1="92" x2="340" y2="130" stroke="#888" stroke-width="1.5"/>
  <polygon points="336,127 344,127 340,138" fill="#888"/>

  <line x1="402" y1="92" x2="565" y2="130" stroke="#888" stroke-width="1.5"/>
  <polygon points="561,126 572,133 570,122" fill="#888"/>

  <!-- Prior box: x=20 to x=205, center=112.5 -->
  <rect x="20" y="133" width="185" height="65" rx="6" fill="#edfaed" stroke="#1a6e2e" stroke-width="2"/>
  <text x="112" y="155" text-anchor="middle" font-size="11" font-weight="bold" fill="#1a6e2e">Class Prior &#x3D5;&#x2C7C;</text>
  <text x="112" y="173" text-anchor="middle" font-size="10" fill="#333">&#x3D5;&#x2C7C; = m&#x2C7C; / m</text>
  <text x="112" y="190" text-anchor="middle" font-size="9" fill="#555">MLE: fraction of class-k samples</text>

  <!-- Means box: x=247 to x=432, center=339.5 -->
  <rect x="247" y="133" width="185" height="65" rx="6" fill="#fff8e8" stroke="#e07b00" stroke-width="2"/>
  <text x="339" y="155" text-anchor="middle" font-size="11" font-weight="bold" fill="#e07b00">Class Means &#x3BC;&#x2C7C;&#x2C7C;</text>
  <text x="339" y="173" text-anchor="middle" font-size="10" fill="#333">&#x3BC;&#x2C7C;&#x2C7C; = mean of feature j in class k</text>
  <text x="339" y="190" text-anchor="middle" font-size="9" fill="#555">MLE: per-class column means</text>

  <!-- Variances box: x=475 to x=660, center=567.5 -->
  <rect x="475" y="133" width="185" height="65" rx="6" fill="#fde8e8" stroke="#c0392b" stroke-width="2"/>
  <text x="567" y="155" text-anchor="middle" font-size="11" font-weight="bold" fill="#c0392b">Class Variances &#x3C3;&#xB2;&#x2C7C;&#x2C7C;</text>
  <text x="567" y="173" text-anchor="middle" font-size="10" fill="#333">&#x3C3;&#xB2;&#x2C7C;&#x2C7C; = var of feature j in class k</text>
  <text x="567" y="190" text-anchor="middle" font-size="9" fill="#555">MLE: per-class column variances</text>

  <!-- Arrows to bottom box -->
  <line x1="112" y1="198" x2="190" y2="248" stroke="#888" stroke-width="1.5"/>
  <line x1="339" y1="198" x2="339" y2="248" stroke="#888" stroke-width="1.5"/>
  <line x1="567" y1="198" x2="490" y2="248" stroke="#888" stroke-width="1.5"/>

  <!-- MAP prediction box -->
  <rect x="80" y="250" width="520" height="60" rx="6" fill="#f0e8fd" stroke="#7b2fc0" stroke-width="2"/>
  <text x="340" y="271" text-anchor="middle" font-size="11" font-weight="bold" fill="#7b2fc0">MAP Prediction</text>
  <text x="340" y="289" text-anchor="middle" font-size="9.5" fill="#444">score&#x2C7C; = log &#x3D5;&#x2C7C; + &#x3A3;&#x2C7C; log N( x&#x2C7C; ; &#x3BC;&#x2C7C;&#x2C7C;, &#x3C3;&#xB2;&#x2C7C;&#x2C7C; )</text>
  <text x="340" y="305" text-anchor="middle" font-size="9.5" fill="#444">&#x177; = argmax&#x2C7C; score&#x2C7C;</text>
</svg>
"""

display(SVG(svg))

## Model / Hypothesis

The MAP decision rule in **log-space** (avoids numerical underflow):

$$\hat{y} = \arg\max_{k} \left[ \log P(y=k) + \sum_{j=1}^{n} \log P(x_j \mid y=k) \right]$$

For **Gaussian NB**, each feature likelihood is:

$$\log P(x_j \mid y=k) = -\tfrac{1}{2}\log(2\pi \sigma^2_{kj}) - \frac{(x_j - \mu_{kj})^2}{2\sigma^2_{kj}}$$

The **naïve independence assumption** factorises the joint likelihood:

$$P(x \mid y=k) = \prod_{j=1}^{n} P(x_j \mid y=k)$$

## Derivation

**Step 1 — Bayes' theorem.**  
The posterior for class $k$ given input $x$ is:

$$P(y=k \mid x) = \frac{P(x \mid y=k)\, P(y=k)}{P(x)}$$

**Step 2 — Drop the evidence.**  
$P(x)$ is the same for all classes, so the argmax is unchanged:

$$\hat{y} = \arg\max_{k}\; P(x \mid y=k)\, P(y=k)$$

**Step 3 — Apply naïve independence.**  
Assume features are conditionally independent given the class:

$$\hat{y} = \arg\max_{k}\; P(y=k) \prod_{j=1}^{n} P(x_j \mid y=k)$$

**Step 4 — Take log (MAP in log-space).**  

$$\hat{y} = \arg\max_{k} \left[ \log P(y=k) + \sum_{j=1}^{n} \log P(x_j \mid y=k) \right]$$

**Step 5 — MLE parameter estimates.**  
Given $m_k$ training examples in class $k$:

$$\phi_k = \frac{m_k}{m}, \qquad \mu_{kj} = \frac{1}{m_k}\sum_{i:\,y^{(i)}=k} x^{(i)}_j, \qquad \sigma^2_{kj} = \frac{1}{m_k}\sum_{i:\,y^{(i)}=k}\bigl(x^{(i)}_j - \mu_{kj}\bigr)^2$$

## Algorithm

**Training — `fit(X, y)`**

1. Find unique classes $\{0, \ldots, K{-}1\}$
2. For each class $k$, collect $X_k = X[y == k]$
3. Compute prior: $\log\phi_k = \log(m_k / m)$
4. Compute means: $\mu_k = \text{mean}(X_k,\; \text{axis}=0)$ — shape $(n,)$
5. Compute variances: $\sigma^2_k = \text{var}(X_k,\; \text{axis}=0) + \varepsilon$ — shape $(n,)$
6. Return `{log_prior, mu, var}` — shapes $(K,)$, $(K, n)$, $(K, n)$

**Prediction — `predict(X, params)`**

1. For each sample, compute log posterior scores for all $K$ classes simultaneously
2. $\text{score}_k = \log\phi_k + \sum_j \log\mathcal{N}(x_j;\; \mu_{kj}, \sigma^2_{kj})$ — vectorised over $(m, K, n)$
3. Return $\hat{y} = \arg\max_k\; \text{score}_k$

---

### Laplace Smoothing (for discrete / count features)

When a feature value never appears in a class during training, its likelihood $P(x_j \mid y=k) = 0$ zeros out the entire product — the **zero-frequency problem**. Laplace smoothing (add-$\alpha$) avoids this by adding a pseudo-count $\alpha$ to every feature-value count before normalising:

$$P(x_j = v \mid y=k) = \frac{\text{count}(x_j = v,\; y=k) + \alpha}{\sum_{v'} \bigl[\text{count}(x_j = v',\; y=k) + \alpha\bigr]}$$

With $\alpha = 1$ (add-one smoothing) no probability is ever exactly zero, and the estimate shrinks smoothly toward a uniform distribution as training data becomes sparse.

> **Note:** Laplace smoothing applies to **Multinomial / Bernoulli NB** (text, categorical features). Gaussian NB uses the variance floor $+\,\varepsilon$ instead.

---

### Shape Reference

| Symbol | Shape | Description |
|---|---|---|
| $X$ | $(m, n)$ | Feature matrix |
| $y$ | $(m,)$ | Integer class labels |
| $\phi$ (`log_prior`) | $(K,)$ | Log class priors |
| $\mu$ | $(K, n)$ | Per-class feature means |
| $\sigma^2$ (`var`) | $(K, n)$ | Per-class feature variances (+ $\varepsilon$) |
| `log_scores` | $(m, K)$ | Log posterior score per sample per class |

## Key Properties

| Property | Detail |
|---|---|
| **Type** | Generative classifier — models $P(x, y)$ |
| **Training** | $O(m \cdot n)$ — one pass per class |
| **Prediction** | $O(K \cdot n)$ per sample |
| **Independence** | Naïve assumption rarely holds exactly, yet accuracy is surprisingly robust |
| **Underflow guard** | Use $\varepsilon = 10^{-9}$ added to variance; work in log-space |
| **Zero-frequency** | For discrete features, Laplace smoothing: add pseudo-count $\alpha=1$ to all counts |
| **Variants** | Gaussian NB (continuous), Multinomial NB (counts/text), Bernoulli NB (binary) |
| **Decision boundary** | Linear when class covariances are equal (reduces to LDA) |

In [ ]:
import numpy as np


def fit(X, y):
    """
    Fit Gaussian Naive Bayes by computing MLE parameters per class.

    Inputs
    ------
    X : ndarray, (m, n)  feature matrix
    y : ndarray, (m,)    integer class labels 0 .. K-1

    Output
    ------
    params : dict
        log_prior : ndarray (K,)    log P(y=k) for each class
        mu        : ndarray (K, n)  per-class feature means
        var       : ndarray (K, n)  per-class feature variances (epsilon-smoothed)
    """
    classes = np.unique(y)
    K = len(classes)
    m, n = X.shape
    mu        = np.zeros((K, n))
    var       = np.zeros((K, n))
    log_prior = np.zeros(K)
    for k in classes:
        Xk          = X[y == k]
        mu[k]       = Xk.mean(axis=0)
        var[k]      = Xk.var(axis=0) + 1e-9      # epsilon avoids log(0)
        log_prior[k] = np.log(len(Xk) / m)
    return dict(log_prior=log_prior, mu=mu, var=var)


def predict_log_proba(X, params):
    """
    Compute log posterior score for each class.

    Inputs
    ------
    X      : ndarray, (m, n)  test features
    params : dict              output of fit()

    Output
    ------
    log_scores : ndarray, (m, K)  unnormalised log posterior per class
    """
    mu  = params['mu']         # (K, n)
    var = params['var']        # (K, n)
    lp  = params['log_prior']  # (K,)
    # Gaussian log-likelihood vectorised over all samples and classes
    # X[:, np.newaxis, :] shape: (m, 1, n) broadcasts against (K, n)
    diff2   = (X[:, np.newaxis, :] - mu) ** 2           # (m, K, n)
    log_lik = -0.5 * (np.log(2 * np.pi * var) + diff2 / var)  # (m, K, n)
    return lp + log_lik.sum(axis=2)                      # (m, K)


def predict(X, params):
    """
    Predict class labels for test samples.

    Inputs
    ------
    X      : ndarray, (m, n)  test features
    params : dict              output of fit()

    Output
    ------
    y_hat : ndarray, (m,)  predicted class labels
    """
    return np.argmax(predict_log_proba(X, params), axis=1)